# 📊 Completeness Monitoring System (OpenSearch + Python)

## 🧠 Overview

This task focuses on building a **data completeness monitoring system** using **OpenSearch** and **Python**.

The goal is to identify:

* Files that are **not completely processed**
* Based on the **latest processing attempt**
* While applying **time-based and logical filters**

---

## 🎯 Objective

To generate a dataset that answers:

> ❓ *How many files are incomplete or missing, based on the latest ingestion?*

---

## 📦 Data Flow

```
Source System → Processing Pipeline → OpenSearch Index
```

Each document in OpenSearch represents a **processed file instance**.

---

## 🔑 Key Fields

| Field               | Description                                             |
| ------------------- | ------------------------------------------------------- |
| `IdFile`            | Unique identifier of a file (can have multiple entries) |
| `TimeCreated`       | Timestamp when file was created at source               |
| `TimeIngest`        | Timestamp when file was ingested into OpenSearch        |
| `CheckCompleteness` | Boolean flag (True = complete, False = incomplete)      |
| `IdProcalc`         | Processing group identifier                             |
| `IdDomainData`      | Sub-group within processing                             |
| `AttrFileSize`      | File size in MB                                         |

---

## ⚠️ Problem Constraints

### 1. Multiple Processing Attempts

* A single `IdFile` may appear multiple times
* Only the **latest record** should be considered
  👉 Based on **maximum `TimeIngest`**

---

### 2. Time-Based Filtering

* Include only documents where:

```
TimeCreated > "2026-02-05T00:00:00Z"
```

---

### 3. Completeness Check

* Only consider:

```
CheckCompleteness = False
```

---

### 4. Ignore Recent Files

* Exclude files that are **less than 2 hours old**
* Reason: pipeline processing may still be ongoing

```
(datetime.now - TimeCreated) > 2 hours
```

---

### 5. Grouping Requirement

Final results should be grouped as:

```
IdProcalc
   └── IdDomainData
         └── List of incomplete files
```

---

## 🏗️ Solution Approach

### Step 1: OpenSearch Query

Use aggregation to:

* Filter relevant documents
* Group by `IdFile`
* Select the **latest document per file**

#### Query Strategy:

* `terms` aggregation → group by `IdFile`
* `top_hits` → fetch latest document using `TimeIngest`

---

### Step 2: Python Processing

Post-process query results to:

* Apply **2-hour exclusion rule**
* Perform **hierarchical grouping**
* Generate **final structured output**

---

## 🔍 Query Logic Summary

| Operation                   | Handled By               |
| --------------------------- | ------------------------ |
| Filter by TimeCreated       | OpenSearch               |
| Filter by CheckCompleteness | OpenSearch               |
| Latest record per IdFile    | OpenSearch (aggregation) |
| 2-hour exclusion            | Python                   |
| Grouping (Procalc → Domain) | Python                   |

---

## 📊 Expected Output Structure

```json
{
  "IdProcalc_1": {
    "IdDomainData_A": [ ...files ],
    "IdDomainData_B": [ ...files ]
  },
  "IdProcalc_2": {
    "IdDomainData_C": [ ...files ]
  }
}
```

---

## 🚦 Status Interpretation (Dashboard)

| Status    | Meaning                         |
| --------- | ------------------------------- |
| 🟢 Green  | Data complete                   |
| 🔴 Red    | Incorrect data (source issue)   |
| 🟠 Orange | Processing issue (system issue) |

---

## ⚙️ Key Design Considerations

* **Aggregation-based deduplication** (latest record selection)
* **Separation of concerns**:

  * Query → filtering & deduplication
  * Python → business logic & grouping
* **Scalable design** (handles multiple records per file)
* **Time-aware filtering** to avoid false positives

---

## 🚀 Future Enhancements

* Add **Prefect flow** for scheduling
* Compute **metrics (counts per group)**
* Integrate with **dashboard visualization**
* Add **alerting for failures**

---

## ✅ Summary

This system ensures:

* Accurate tracking of incomplete files
* Reliable monitoring using latest data
* Clean separation between query and processing logic

---
